In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import shap

# ======================== User configuration ========================
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
OUT_BASE = Path("../outputs/shap")
RANDOM_STATE = 42
N_SPLITS = 5
# ==========================================================

KEY = ["city", "cell_id", "year"]
TBL = OUT_BASE / "tables"
TBL.mkdir(parents=True, exist_ok=True)


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


# ============================================================
# 1. Read data and samples
# ============================================================
require(PANEL.is_file(), f"Panel not found: {PANEL}")
raw = pd.read_csv(PANEL)
require(len(raw) == 5216 and not raw.duplicated(KEY).any(), "Abnormal panel row count or keys")
require(raw.pct.eq("p90").all(), "The local_p90 main panel should be used here")

MORPH_Z = ["bld_coverage_z", "bld_mean_h_m_z", "bld_std_h_m_z", "lcz_1_z", "lcz_2_z", "lcz_3_z",
           "ndvi_mean_z", "water_coverage_z", "isa_mean_prob_z"]
CLIMATE_FEATS = ["cdd_22C_city", "rh_mean_pct", "tp_annual_mm", "lat", "dem_mean_m", "coast_dist_km"]
CAT_FEATS = ["koppen_B", "koppen_C", "region_SA", "region_SEA", "region_LA"]
FEAT_EXTRA = ["year_num"]
ALL_FEATS = MORPH_Z + CLIMATE_FEATS + CAT_FEATS + FEAT_EXTRA
OUTCOMES = ["log_exp_day", "log_exp_night", "log_exp_compound"]
FEATURE_LABELS = {"bld_coverage_z": "Building density", "bld_mean_h_m_z": "Mean building height",
                  "bld_std_h_m_z": "Building height SD", "lcz_1_z": "LCZ 1 (comp hi)",
                  "lcz_2_z": "LCZ 2 (comp mid)", "lcz_3_z": "LCZ 3 (comp lo)", "ndvi_mean_z": "NDVI",
                  "water_coverage_z": "Water coverage", "isa_mean_prob_z": "Impervious cover",
                  "cdd_22C_city": "CDD (22°C)", "rh_mean_pct": "Humidity", "tp_annual_mm": "Precip",
                  "lat": "Latitude", "dem_mean_m": "Elevation", "coast_dist_km": "Coast dist",
                  "koppen_B": "Köppen B", "koppen_C": "Köppen C",
                  "region_SA": "Region SA", "region_SEA": "Region SEA", "region_LA": "Region LA",
                  "year_num": "Year"}

base_flag = raw.heatwave_annual_complete.astype(str).str.lower().eq("true") & \
            raw.eligible_density_model.astype(str).str.lower().eq("true")
mask = raw.sample_ml_controls.astype(str).str.lower().eq("true")
require((mask == (base_flag & raw[ALL_FEATS + OUTCOMES].notna().all(axis=1))).all(),
        "sample_ml_controls is inconsistent with feature completeness")
df = raw.loc[mask].copy().sort_values(KEY).reset_index(drop=True)
require(df[ALL_FEATS + OUTCOMES].notna().all().all(), "Sample still has missing values")
df[KEY + ["koppen_broad"] + ALL_FEATS].to_csv(TBL / "ml_sample_features.csv", index=False, float_format="%.17g")

# ============================================================
# 2. Arrays and cross-validation splits
# ============================================================
X_df = df[ALL_FEATS].astype(float).copy()
X_all = X_df.values
cell_groups = (df.city.astype(str) + "|" + df.cell_id.astype(str)).values

CV_SCHEMES = {
    "kf": list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X_all)),
    "cellkf": list(GroupKFold(n_splits=N_SPLITS).split(X_all, groups=cell_groups)),
}


# ============================================================
# 3. XGBoost
# ============================================================
def make_xgb():
    return xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8,
                            colsample_bytree=0.8, tree_method="hist", random_state=RANDOM_STATE, verbosity=0)


def cv_metrics(model, X, y, splits):
    r2s, rmses, maes = [], [], []
    for tr, te in splits:
        m = clone(model)
        m.fit(X[tr], y[tr])
        yp = m.predict(X[te])
        r2s.append(r2_score(y[te], yp))
        rmses.append(np.sqrt(mean_squared_error(y[te], yp)))
        maes.append(mean_absolute_error(y[te], yp))
    return np.mean(r2s), np.std(r2s), np.mean(rmses), np.std(rmses), np.mean(maes), np.std(maes)


# ============================================================
# 4. Cross-validation
# ============================================================
cv_records = []
for out in OUTCOMES:
    y = df[out].values
    rec, msg = {"outcome": out, "model": "XGBoost"}, f"  {out:18s}"
    for key, splits in CV_SCHEMES.items():
        r2, r2s, rmse, rmses, mae, maes = cv_metrics(make_xgb(), X_all, y, splits)
        rec.update({f"{key}_R2": r2, f"{key}_R2_std": r2s, f"{key}_RMSE": rmse, f"{key}_RMSE_std": rmses,
                    f"{key}_MAE": mae, f"{key}_MAE_std": maes})
        msg += f"  {key} R²={r2:+.3f}"
    cv_records.append(rec)
    print(msg)
pd.DataFrame(cv_records).to_csv(TBL / "ml_cv_metrics_xgboost.csv", index=False)

# ============================================================
# 5. Full-sample fit and SHAP
# ============================================================
def compute_shap_values(model, X_df):
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_df.values)
    except Exception:
        booster = model.get_booster()
        booster.feature_names = list(X_df.columns)
        explainer = shap.TreeExplainer(booster)
        sv = explainer.shap_values(X_df.values)
    base_value = np.ravel(explainer.expected_value)[0]
    err = np.max(np.abs(sv.sum(axis=1) + base_value - model.predict(X_df.values)))
    require(err < 1e-3, f"SHAP additivity check failed (max error {err:.2e})")
    return sv


for out in OUTCOMES:
    y = df[out].values
    model = make_xgb()
    model.fit(X_all, y)
    sv = compute_shap_values(model, X_df)
    pd.concat([df[KEY], pd.DataFrame(sv, columns=ALL_FEATS)], axis=1) \
        .to_csv(TBL / f"shap_values_xgboost_{out}.csv", index=False, float_format="%.17g")
    pd.DataFrame({"feature": ALL_FEATS, "feature_label": [FEATURE_LABELS[f] for f in ALL_FEATS],
                  "mean_abs_shap": np.abs(sv).mean(axis=0)}) \
        .sort_values("mean_abs_shap", ascending=False) \
        .to_csv(TBL / f"shap_importance_xgboost_{out}.csv", index=False)
    print(f"  {out}: train R²={r2_score(y, model.predict(X_all)):.3f}; SHAP finished (additivity check passed)")

(OUT_BASE / "configuration.json").write_text(json.dumps(dict(
    panel=str(PANEL), n=len(df), n_cities=int(df.city.nunique()),
    model="XGBoost", hyperparameters=make_xgb().get_params(), features=ALL_FEATS, outcomes=OUTCOMES,
    random_state=RANDOM_STATE, n_splits=N_SPLITS,
    shap=dict(version=shap.__version__, xgboost=xgb.__version__)),
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Output folder:", OUT_BASE)